# Adım 4: Keşifsel Veri Analizi (EDA)

Delta Lake Silver katmanına yazılan NYC Taxi verisi üzerinde kapsamlı EDA gerçekleştirilir.

| Bölüm | Konu |
|-------|------|
| 1-3 | Kurulum & veri yükleme |
| 4-5 | Temel istatistikler & eksik değer analizi |
| 6-7 | Zaman serisi (günlük/saatlik trendler) |
| 8-9 | Kategorik & sayısal dağılım analizi |
| 10 | Korelasyon matrisi |
| 11 | Özet bulgular |

## 1. Kütüphane Kurulumu

In [ ]:
import subprocess, sys
pkgs = ["pandas", "numpy", "matplotlib", "seaborn", "pyarrow==15.0.2"]
for pkg in pkgs:
    r = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                       capture_output=True, text=True)
    print(f"{'✅' if r.returncode==0 else '❌'} {pkg}")

## 2. Import & Görsel Ayarlar

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})

PROJECT_ROOT = Path(os.getcwd()).parent
SILVER_PATH  = PROJECT_ROOT / "data/delta/silver/yellow_taxi_trips"
PARQUET_PATH = PROJECT_ROOT / "data/raw/yellow_tripdata_2023-01.parquet"

print("Kütüphaneler yüklendi.")
print(f"Pandas  : {pd.__version__}")
print(f"NumPy   : {np.__version__}")

## 3. Veri Yükleme

Silver Delta yoksa ham Parquet'ten yükler:

In [ ]:
import pyarrow.parquet as pq

SAMPLE_SIZE = 50_000

silver_parquets = list(SILVER_PATH.rglob("*.parquet")) if SILVER_PATH.exists() else []

if silver_parquets:
    print(f"Silver Delta: {len(silver_parquets)} parquet dosyası")
    df = pd.read_parquet(SILVER_PATH, engine="pyarrow")
    kaynak = "Silver Delta"
elif PARQUET_PATH.exists():
    print("Silver bulunamadı → ham Parquet kullanılıyor")
    raw = pq.read_table(PARQUET_PATH).to_pandas()
    raw["pickup_datetime"]    = pd.to_datetime(raw["tpep_pickup_datetime"], errors="coerce")
    raw["dropoff_datetime"]   = pd.to_datetime(raw["tpep_dropoff_datetime"], errors="coerce")
    raw["trip_duration_minutes"] = (
        raw["dropoff_datetime"] - raw["pickup_datetime"]
    ).dt.total_seconds() / 60
    raw["pickup_hour"]        = raw["pickup_datetime"].dt.hour
    raw["pickup_day_of_week"] = raw["pickup_datetime"].dt.dayofweek + 1
    raw["pickup_date"]        = raw["pickup_datetime"].dt.date
    raw["pickup_month"]       = raw["pickup_datetime"].dt.month
    raw["pickup_year"]        = raw["pickup_datetime"].dt.year
    df = raw[
        (raw["trip_distance"] > 0) & (raw["trip_distance"] <= 100) &
        (raw["fare_amount"]   > 0) & (raw["fare_amount"]   <= 500) &
        (raw["trip_duration_minutes"] > 0) & (raw["trip_duration_minutes"] <= 240)
    ].copy()
    kaynak = "Ham Parquet"
else:
    raise FileNotFoundError("Ne Silver Delta ne de ham Parquet bulunamadı!")

if len(df) > SAMPLE_SIZE:
    df = df.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    print(f"Örnekleme: {SAMPLE_SIZE:,} satır")

print(f"\nKaynak   : {kaynak}")
print(f"Satır    : {len(df):,}")
print(f"Sütun    : {len(df.columns)}")

## 4. Temel İstatistikler

In [ ]:
sayisal = ["fare_amount", "trip_distance", "trip_duration_minutes",
           "passenger_count", "tip_amount", "total_amount"]
mevcut  = [c for c in sayisal if c in df.columns]

istat = df[mevcut].describe().T
istat["cv%"] = (istat["std"] / istat["mean"] * 100).round(1)
print("=== Temel İstatistikler ===")
print(istat.round(2).to_string())

## 5. Eksik Değer Analizi

In [ ]:
eksik = df.isnull().sum()
eksik = eksik[eksik > 0].sort_values(ascending=False)

if eksik.empty:
    print("✅ Hiç eksik değer yok!")
else:
    oran = (eksik / len(df) * 100).round(2)
    tablo = pd.DataFrame({"Eksik Sayı": eksik, "Oran %": oran})
    print(tablo.to_string())

    fig, ax = plt.subplots(figsize=(8, max(3, len(eksik)*0.5)))
    oran.plot.barh(ax=ax, color="#e74c3c")
    ax.set_xlabel("Eksik Değer Oranı (%)")
    ax.set_title("Sütun Bazında Eksik Değer Oranları")
    plt.tight_layout()
    plt.show()

## 6. Zaman Serisi: Günlük Trip Trendi

In [ ]:
if "pickup_date" in df.columns:
    gunluk = df.groupby("pickup_date").size().reset_index(name="trip_count")
    gunluk["pickup_date"] = pd.to_datetime(gunluk["pickup_date"])

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(gunluk["pickup_date"], gunluk["trip_count"],
            color="#3498db", linewidth=2)
    ax.fill_between(gunluk["pickup_date"], gunluk["trip_count"],
                    alpha=0.15, color="#3498db")
    ax.set_title("Günlük Yolculuk Sayısı")
    ax.set_xlabel("Tarih")
    ax.set_ylabel("Trip Sayısı")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    plt.tight_layout()
    plt.show()

## 7. Zaman Serisi: Saatlik & Haftanın Günü Dağılımı

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Saatlik dağılım
if "pickup_hour" in df.columns:
    saatlik = df.groupby("pickup_hour").size()
    axes[0].bar(saatlik.index, saatlik.values, color="#9b59b6", alpha=0.85)
    axes[0].set_title("Saatlik Trip Dağılımı")
    axes[0].set_xlabel("Saat")
    axes[0].set_ylabel("Trip Sayısı")
    axes[0].set_xticks(range(0, 24, 2))

# Haftanın günü dağılımı
if "pickup_day_of_week" in df.columns:
    gun_isimleri = {1:"Paz",2:"Pzt",3:"Sal",4:"Çar",5:"Per",6:"Cum",7:"Cmt"}
    gunluk2 = df.groupby("pickup_day_of_week").size()
    gunluk2.index = gunluk2.index.map(gun_isimleri)
    axes[1].bar(gunluk2.index, gunluk2.values, color="#e67e22", alpha=0.85)
    axes[1].set_title("Haftanın Gününe Göre Trip Dağılımı")
    axes[1].set_xlabel("Gün")
    axes[1].set_ylabel("Trip Sayısı")

plt.tight_layout()
plt.show()

## 8. Sayısal Değişken Dağılımları

In [ ]:
degiskenler = [
    ("fare_amount",           "Ücret ($)",         (0, 100)),
    ("trip_distance",         "Mesafe (mil)",       (0, 30)),
    ("trip_duration_minutes", "Süre (dk)",          (0, 120)),
    ("tip_amount",            "Bahşiş ($)",         (0, 30)),
]
mevcut_deg = [(c, l, r) for c, l, r in degiskenler if c in df.columns]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, (col_name, label, xlim) in enumerate(mevcut_deg):
    veri = df[col_name].dropna()
    veri = veri[(veri >= xlim[0]) & (veri <= xlim[1])]
    axes[i].hist(veri, bins=50, color="#1abc9c", alpha=0.8, edgecolor="white")
    axes[i].axvline(veri.mean(), color="red", linestyle="--", linewidth=1.5,
                    label=f"Ort: {veri.mean():.2f}")
    axes[i].axvline(veri.median(), color="orange", linestyle="--", linewidth=1.5,
                    label=f"Med: {veri.median():.2f}")
    axes[i].set_title(label)
    axes[i].set_xlabel(label)
    axes[i].set_ylabel("Frekans")
    axes[i].legend(fontsize=9)

for j in range(len(mevcut_deg), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Sayısal Değişken Dağılımları", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 9. Kategorik Değişken Analizi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Borough dağılımı
if "pickup_borough" in df.columns:
    borough_counts = df["pickup_borough"].value_counts()
    axes[0].barh(borough_counts.index, borough_counts.values, color="#2980b9", alpha=0.85)
    axes[0].set_title("Biniş İlçesi (Borough) Dağılımı")
    axes[0].set_xlabel("Trip Sayısı")
    for i, v in enumerate(borough_counts.values):
        axes[0].text(v + borough_counts.max()*0.01, i, f"{v:,}", va="center", fontsize=9)

# Payment type
if "payment_type" in df.columns:
    odeme = {1:"Kredi Kartı", 2:"Nakit", 3:"Ücretsiz", 4:"Anlaşmazlık", 5:"Bilinmiyor", 6:"Void"}
    pt = df["payment_type"].map(odeme).fillna("Diğer").value_counts()
    axes[1].pie(pt.values, labels=pt.index, autopct="%1.1f%%",
                colors=sns.color_palette("husl", len(pt)))
    axes[1].set_title("Ödeme Tipi Dağılımı")

plt.tight_layout()
plt.show()

### Saate Göre Ortalama Ücret

In [ ]:
if "pickup_hour" in df.columns and "fare_amount" in df.columns:
    saat_ucret = df.groupby("pickup_hour")["fare_amount"].agg(["mean","median","count"])

    fig, ax1 = plt.subplots(figsize=(12, 5))
    ax2 = ax1.twinx()

    ax1.bar(saat_ucret.index, saat_ucret["count"], color="#bdc3c7", alpha=0.5, label="Trip sayısı")
    ax2.plot(saat_ucret.index, saat_ucret["mean"],   "o-", color="#e74c3c", linewidth=2, label="Ort. Ücret")
    ax2.plot(saat_ucret.index, saat_ucret["median"], "s--", color="#3498db", linewidth=2, label="Medyan Ücret")

    ax1.set_xlabel("Saat")
    ax1.set_ylabel("Trip Sayısı")
    ax2.set_ylabel("Ücret ($)")
    ax1.set_xticks(range(0, 24))

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
    ax1.set_title("Saate Göre Trip Sayısı ve Ortalama Ücret")
    plt.tight_layout()
    plt.show()

## 10. Korelasyon Matrisi

In [ ]:
korelasyon_sutunlar = [
    "fare_amount", "trip_distance", "trip_duration_minutes",
    "passenger_count", "tip_amount", "pickup_hour", "pickup_day_of_week"
]
mevcut_kor = [c for c in korelasyon_sutunlar if c in df.columns]

corr = df[mevcut_kor].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f",
            cmap="RdYlGn", center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax)
ax.set_title("Değişkenler Arası Korelasyon Matrisi", pad=15)
plt.tight_layout()
plt.show()

# En yüksek korelasyonlar
kor_uzun = corr.where(~mask).stack().sort_values(ascending=False)
print("\nEn yüksek 5 korelasyon:")
for (a, b), v in kor_uzun.head(5).items():
    print(f"  {a:<28} ↔ {b:<28} {v:.3f}")

## 11. Özet Bulgular & Teslim Edilecekler

In [ ]:
print("=" * 55)
print("  EDA ÖZET BULGULARI")
print("=" * 55)
print(f"  Analiz edilen satır sayısı  : {len(df):>10,}")
if "fare_amount" in df.columns:
    print(f"  Ortalama ücret ($)          : {df['fare_amount'].mean():>10.2f}")
    print(f"  Medyan ücret ($)            : {df['fare_amount'].median():>10.2f}")
if "trip_distance" in df.columns:
    print(f"  Ortalama mesafe (mil)       : {df['trip_distance'].mean():>10.2f}")
if "trip_duration_minutes" in df.columns:
    print(f"  Ortalama süre (dk)          : {df['trip_duration_minutes'].mean():>10.2f}")
if "pickup_hour" in df.columns:
    en_yogun_saat = df['pickup_hour'].value_counts().idxmax()
    print(f"  En yoğun saat               : {en_yogun_saat:>10}:00")
if "pickup_borough" in df.columns:
    en_yogun_ilce = df['pickup_borough'].value_counts().idxmax()
    print(f"  En yoğun ilçe               : {en_yogun_ilce:>10}")
print("=" * 55)

print("""
Teslim Edilecekler:
  ✅  Temel istatistikler tablosu
  ✅  Eksik değer analizi ve görselleştirme
  ✅  Günlük / saatlik / haftanın günü trend grafikleri
  ✅  Sayısal değişken dağılım histogramları
  ✅  Kategorik (borough, payment_type) pasta/bar grafikler
  ✅  Saate göre ücret & yoğunluk analizi
  ✅  Korelasyon matrisi (heatmap)
""")

---
> **Sonraki adım →** Adım 5: Makine Öğrenmesi Modeli (MLflow ile)